# FIREGUARD ID v2 — Minggu 2: [L] Learning Problem, Data Understanding & EDA
**Sesuai Ketentuan Minggu 1–2:** Identifikasi masalah SDG 13.1, audit dataset, missing values/duplikat/outlier, serta >=3 visualisasi EDA & insight.

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import json
import numpy as np
import pandas as pd
from src.utils import load_yaml, resolve_path

proj_cfg = load_yaml("config/project.yaml", repo_root=REPO_ROOT)
print("Repo root:", REPO_ROOT)

Repo root: FIREGUARD-ID


## 1. Audit Sumber Data & Provenance (`SRC-01` s.d. `SRC-10`)

In [2]:
src_df = pd.read_csv(resolve_path("reports/data_quality/data_source_status.csv", REPO_ROOT))
display(src_df[["source_id", "dataset_name", "status", "observation_period", "spatial_resolution"]])

qa = json.loads(resolve_path("reports/data_quality/data_quality_summary.json", REPO_ROOT).read_text(encoding="utf-8"))
print("Raw VIIRS N20 rows:", qa["quality_audit"]["raw_record_count"])
print("Riau-clipped rows:", qa["quality_audit"]["clipped_record_count"])
print("Riau geodesic area (km2):", qa["boundary"]["geodesic_area_km2"])
print("Full-row duplicates (clipped):", qa["quality_audit"]["duplicates"]["clipped_full_row"])

,source_id,dataset_name,status,observation_period,spatial_resolution
0,SRC-01,NASA FIRMS VIIRS NOAA-20 (JPSS-1),AVAILABLE,2021-01-01 to 2025-12-31,375 m nadir pixel
1,SRC-02,GADM 4.1 Indonesia Level-1 Boundary (Riau),AVAILABLE,Static v4.1,"Vector MultiPolygon (73 parts, 1 interior hole)"
2,SRC-03,NASA FIRMS VIIRS S-NPP Archive (2012-2025),UNAVAILABLE,2012-01-20 to 2025-12-31 (target),375 m
3,SRC-04,Copernicus ERA5-Land Daily Meteorology & Soil ...,AVAILABLE,2021-01-01 to 2025-12-31,0.75 deg anchor grid (26 anchors) IDW-joined t...
4,SRC-05,CHIRPS v2.0 Monthly/Daily Precipitation,UNAVAILABLE,1981-present (target),0.05 deg (~5.5 km)
5,SRC-06,MODIS MOD13Q1 (NDVI/EVI) & MCD12Q1 (Land Cover),UNAVAILABLE,2000-present (target),250 m (MOD13Q1) / 500 m (MCD12Q1)
6,SRC-07,Copernicus DEM 90m Terrain Elevation (DEMNAS A...,AVAILABLE,Static,90 m sampled at 0.1-deg cell representative la...
7,SRC-08,Authoritative Indonesian Peatland Distribution,UNAVAILABLE,Static / Periodic,Vector / 30m-250m
8,SRC-09,OpenStreetMap Emergency Fire Stations (Overpass),UNAVAILABLE,Current snapshot (2026-11),Point / Way vectors
9,SRC-10,BPBD / Manggala Agni Emergency Facility & Flee...,SYNTHETIC,Planning scenario,Facility / District level


Raw VIIRS N20 rows: 595676
Riau-clipped rows: 15510
Riau geodesic area (km2): 89874.5
Full-row duplicates (clipped): 0


## 2. Ringkasan EDA (Tren Tahunan, Siklus Musiman, Distribusi Grid & Moran's I)

In [3]:
eda_md = resolve_path("reports/eda/eda_summary.md", REPO_ROOT).read_text(encoding="utf-8")
print(eda_md)

# FIREGUARD ID — Exploratory Data Analysis Summary (Phase 4)

**Dataset Scope:** 15,510 Riau-clipped NOAA-20 VIIRS detections across 853 intersecting 0.1° cells (2021–2025).

## Key Analytical Findings
1. **Inter-Annual Variability:** Annual counts peak in 2023 (`3,308`) and 2025 (`4,004`), with lower totals in 2021–2022 and 2024.
2. **Bimodal Dry-Season Concentration:** Detections surge in Feb–Mar and Jul–Sep (peak monthly mean in month `7`), consistent with Riau's equatorial monsoon regime.
3. **Strong Spatial Skew & Zero-Detection Cells:** 249 of 853 cells (29.2%) recorded zero detections in 2021–2025, while chronic multi-year cells (active in 4–5 years) account for persistent activity.
4. **Significant Positive Spatial Autocorrelation:** Global Moran's I for `log1p(hotspot_density_km2)` across queen-adjacent cells is **0.2020**, confirming strong geographic clustering of fire activity.
5. **QA Metadata Separation:** High-confidence detections represent 2.3% (`361 / 15,510`), nomina